# Notebook 5: Ownership Projection Model

**Goal:** Predict DraftKings GPP ownership % for contrarian leverage.

**Why this matters:** Ownership projections determine contrarian plays. If you know a player will be 30% owned, you can fade them in GPPs for leverage.

**Input:** `yakos_features.parquet` + historical actual ownership data (RG exports / DK contest CSVs)

**Output:** `models/yakos_ownership_model.pkl`

## Evaluation Targets
- MAE ≤ 3% ownership
- Correct rank order even if exact % is off

## Data Note
Training requires historical actual ownership from:
- RG contest exports with actual ownership columns
- DraftKings contest results CSVs
- ResultsDB or similar DFS results aggregators

In [ ]:
# !pip install scikit-learn lightgbm joblib pyarrow -q

import os
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

INPUT_PATH = 'yakos_features.parquet'
MODEL_DIR  = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(INPUT_PATH)
df['game_date'] = pd.to_datetime(df['game_date'])
print(f'Features shape: {df.shape}')

## Load Actual Ownership Data

Merge actual ownership from RotoGrinders exports or DK contest CSVs if available. Otherwise we fall back to RG projected ownership.

In [ ]:
def load_actual_ownership(rg_dir='.'):
    """Load actual ownership from RotoGrinders contest export CSVs."""
    from pathlib import Path
    all_dfs = []
    for csv_path in Path(rg_dir).glob('NBADK*.csv'):
        df_rg = pd.read_csv(csv_path)
        # Normalise column names
        df_rg.columns = [c.strip() for c in df_rg.columns]
        name_col = next((c for c in df_rg.columns if c.lower() in ('name', 'player')), None)
        own_col  = next((c for c in df_rg.columns if 'own' in c.lower()), None)
        if name_col and own_col:
            tmp = df_rg[[name_col, own_col]].rename(
                columns={name_col: 'player_name', own_col: 'actual_ownership'}
            )
            stem = csv_path.stem
            digits = ''.join(c for c in stem if c.isdigit())
            if len(digits) >= 8:
                d = digits[:8]
                tmp['game_date'] = f'{d[:4]}-{d[4:6]}-{d[6:]}'
            all_dfs.append(tmp)
    if not all_dfs:
        return pd.DataFrame()
    return pd.concat(all_dfs, ignore_index=True)


actual_own = load_actual_ownership(rg_dir='.')

# Merge actual ownership into features
if len(actual_own) > 0 and 'game_date' in actual_own.columns:
    actual_own['game_date'] = pd.to_datetime(actual_own['game_date'])
    df = df.merge(actual_own, on=['player_name', 'game_date'], how='left')
    TARGET = 'actual_ownership'
    print(f'Actual ownership merged: {df["actual_ownership"].notna().sum()} rows with target')
elif 'rg_ownership' in df.columns:
    TARGET = 'rg_ownership'
    print('Using rg_ownership as proxy target (actual ownership not available)')
else:
    # Generate synthetic ownership target from value score for demo
    print('No ownership data available. Generating synthetic target from value score.')
    df['synthetic_own'] = (df.get('value_score', df['salary'] / 10000.0) - 2.0) * 8.0
    df['synthetic_own'] = df['synthetic_own'].clip(0, 50)
    TARGET = 'synthetic_own'

## Train Ownership Model

In [ ]:
CUTOFF_DATE = pd.Timestamp('2026-02-15')

OWN_FEATURE_COLS = [
    'salary', 'tank01_proj', 'rg_proj',
    'rolling_fp_5', 'rolling_fp_10',
    'value_score', 'trend',
    'b2b', 'dvp',
]

available_feats = [c for c in OWN_FEATURE_COLS if c in df.columns]
print(f'Features: {available_feats}')

model_df = df.dropna(subset=[TARGET])
train_df = model_df[model_df['game_date'] < CUTOFF_DATE]
val_df   = model_df[model_df['game_date'] >= CUTOFF_DATE]

X_train = train_df[available_feats]
y_train = pd.to_numeric(train_df[TARGET], errors='coerce')
X_val   = val_df[available_feats]
y_val   = pd.to_numeric(val_df[TARGET], errors='coerce')

# Drop NaN targets
valid_train = y_train.notna()
valid_val   = y_val.notna()
X_train, y_train = X_train[valid_train], y_train[valid_train]
X_val, y_val     = X_val[valid_val], y_val[valid_val]

print(f'Train: {len(X_train)} | Val: {len(X_val)}')

ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=5.0)),
])

if len(X_train) > 0:
    ridge_pipeline.fit(X_train, y_train)
    preds = ridge_pipeline.predict(X_val)
    preds = np.clip(preds, 0, 100)

    mae  = mean_absolute_error(y_val, preds)
    bias = (preds - y_val).mean()
    rank_corr, _ = spearmanr(y_val, preds)

    print('=== Ownership Model ===')
    print(f'  MAE:           {mae:.2f}%  (target ≤ 3%)')
    print(f'  Bias:          {bias:+.2f}%')
    print(f'  Rank Corr:     {rank_corr:.3f}  (higher = better rank order)')
else:
    print('Insufficient training data. Model not fitted.')

## Save Ownership Model

In [ ]:
if len(X_train) > 0:
    model_path = MODEL_DIR / 'yakos_ownership_model.pkl'
    joblib.dump(ridge_pipeline, model_path)
    print(f'Ownership model saved: {model_path}')

    with open(MODEL_DIR / 'yakos_ownership_features.json', 'w') as f:
        json.dump(available_feats, f)

    metrics = {
        'target': TARGET,
        'mae': float(mae),
        'bias': float(bias),
        'rank_correlation': float(rank_corr),
    }
    with open(MODEL_DIR / 'yakos_ownership_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    print('Metrics:', metrics)
else:
    print('Skipped: no training data.')